# 基本構文：イベント列から関数とコマンドへ

対応する本文: [basics.tex](../chapters/basics.tex)。各問を「考え方 → 実装 → 確認」の順で解説する。上から全セルを実行すると、入力の作成から結果の検証までを再現できる。生成物は専用の出力先へ保存する。

In [1]:
import sys
from pathlib import Path

# リポジトリ直下と notebooks/ のどちらから起動しても同じ場所を使う。
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "main.tex").exists() and (p / "chapters").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("教材のフォルダー内で Notebook を起動してください。")
WORK = ROOT / "notebooks" / "_outputs" / "03_basics"
WORK.mkdir(parents=True, exist_ok=True)
print("Python", sys.version.split()[0])
print("出力先:", WORK.relative_to(ROOT))

Python 3.12.14
出力先: notebooks/_outputs/03_basics


In [2]:
events = [
    {"detector": "A", "time_ns": 1000, "signal": 12.4},
    {"detector": "B", "time_ns": 1180, "signal": 9.8},
    {"detector": "A", "time_ns": 1300, "signal": 15.1},
    {"detector": "B", "time_ns": 1510, "signal": None},
]

## 問1：欠損を判定する

`None` は値がないことを表す。`if signal` では有効値の `0.0` まで除いてしまうため、`is not None` で判定する。辞書のキーは値を取り出す名前である。

In [3]:
lines = []
for event in events:
    if event["signal"] is not None:
        lines.append(f'{event["detector"]} {event["time_ns"]} {event["signal"]}')
print("\n".join(lines))
assert len(lines) == 3 and lines[0] == "A 1000 12.4"

A 1000 12.4
B 1180 9.8
A 1300 15.1


## 問2：平均を関数にする

検出器名と有効値の両方で絞る。空の入力に対して0を返すと測定値の0と区別できないため、例外で知らせる。

In [4]:
def mean_signal(events, detector):
    """指定した検出器の有効信号の算術平均を返す。

    Args:
        events: detector と signal を持つ辞書の列。
        detector: 集計する検出器名。
    Returns:
        None を除いた信号値の算術平均。
    Raises:
        ValueError: 該当する有効値がない場合。
    """
    values = [e["signal"] for e in events
              if e["detector"] == detector and e["signal"] is not None]
    if not values:
        raise ValueError("有効な信号値がありません。")
    return sum(values) / len(values)

print(mean_signal(events, "A"), mean_signal(events, "B"))
assert mean_signal(events, "A") == 13.75
assert mean_signal(events, "B") == 9.8
assert mean_signal([{"detector": "A", "signal": 0.0}], "A") == 0.0
try:
    mean_signal(events, "C")
except ValueError as exc:
    print(type(exc).__name__, str(exc))
else:
    raise AssertionError("空入力の例外がありません。")

13.75 9.8
ValueError 有効な信号値がありません。


## 問3：隣接差を計算する

4個の時刻からは3個の差を作る。`range(len(times) - 1)` は `0, 1, 2` を返し、最後の `i + 1` は3となる。`range` の終了値自体は含まれない。

In [5]:
times = sorted([e["time_ns"] for e in events])
gaps = [times[i + 1] - times[i] for i in range(len(times) - 1)]
print(times, gaps)
assert gaps == [180, 120, 210]

[1000, 1180, 1300, 1510] [180, 120, 210]


## 問4：位置引数とオプション引数を使う

検出器名は必須の位置引数、`--min-signal` は省略できるオプションである。コードをファイルへ保存し、カーネルと同じ Python で別プロセスとして起動する。

In [6]:
import subprocess

script = WORK / "filter_events.py"
script.write_text("import argparse\n" + "events = " + repr(events) + "\n" +
    "parser = argparse.ArgumentParser()\n"
    "parser.add_argument('detector')\n"
    "parser.add_argument('--min-signal', type=float, default=0.0)\n"
    "args = parser.parse_args()\n"
    "for event in events:\n"
    "    if (event['detector'] == args.detector and event['signal'] is not None\n"
    "            and event['signal'] >= args.min_signal):\n"
    "        print(event['detector'], event['time_ns'], event['signal'])\n", encoding="utf-8")
result = subprocess.run([sys.executable, str(script), "A", "--min-signal", "13"], capture_output=True, text=True, check=True)
print(result.stdout)
assert result.stdout.strip() == "A 1300 15.1"

A 1300 15.1

